In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#==========================================================
# Load Packages 
#==========================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import numpy as np, torch
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
import sys
sys.path.append('/project/IZZY/MolRepres/Methods/')                  # Path where the training, evaluating function is located 
from torch_geometric.datasets import QM9                             # Dataset utilities for 3D molecular graphs (QM9 with 3D coordinates)
from torch_geometric.nn import global_mean_pool, GINConv             # Graph Neural Network layers and pooling operations from PyTorch Geometric
from torch_geometric.loader import DataLoader                        # DataLoader to batch and shuffle molecular graph data
from torch.optim import Adam                                         # Optimizer (Adam) and learning rate scheduler (StepLR)
from torch.optim.lr_scheduler import StepLR                          # Progress bar for training/evaluation loops
from tqdm import tqdm                                                # Unit conversion constants (Hartree, eV, Bohr, Angstrom) from ASE
from ase.units import Hartree, eV, Bohr, Ang                         # TensorBoard writer for tracking metrics and visualizing training progress
from torch.utils.tensorboard import SummaryWriter                    # TensorBoard writer for tracking metrics and visualizing training progress
from train_eval import train_epoch, evaluate                         # Importing the training and evaluating function
from sklearn.preprocessing import StandardScaler                     # Tool for normalizing data (zero mean, unit variance)

NumPy: 1.26.4
Torch: 2.0.1+cu118


In [3]:
#==========================================================
# GPU Device
#==========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # This line checks if a CUDA-enabled GPU is available. If yes, computations will be performed on the GPU for faster training. Otherwise, it falls back to using the CPU.
# Print which device is being used 
print("Using device:", device)

Using device: cuda


In [4]:
#==============================================================
# Load dataset and split
#==============================================================

# Load the QM9 dataset 
dataset = QM9(root='data/QM9')

# Count how many molecular graphs are in the dataset
num_mols = len(dataset)
print(f"Total QM9 molecules: {num_mols}")

# Define the desired split sizes for training, validation, and testing
train_size, valid_size = 110000, 10000
# Ensure the sum of train and validation sizes does not exceed dataset size
assert train_size + valid_size < num_mols, "Split sizes too large for dataset"

# Randomly shuffle all molecule indices
perm = torch.randperm(num_mols, generator=torch.Generator().manual_seed(42))

# Use the shuffled indices to create non-overlapping subsets
train_idx = perm[:train_size]
valid_idx = perm[train_size:train_size+valid_size]
test_idx  = perm[train_size+valid_size:]

# Build subsets based on the index splits
train_dataset = dataset[train_idx]
valid_dataset = dataset[valid_idx]
test_dataset  = dataset[test_idx]

Total QM9 molecules: 130831


In [5]:
#======================================================================
# Define the QM9 Targerts
#======================================================================
PROPERTY_NAMES = [
    'μ (D)', 'α (Ang³)', 'ε_HOMO (eV)', 'ε_LUMO (eV)', 'Δε (eV)', '⟨R²⟩ (Ang²)',
    'ZPVE (eV)', 'U₀ (eV)', 'U (eV)', 'H (eV)', 'G (eV)', 'c_v', 'U₀_atom',
    'U_atom', 'H_atom', 'G_atom', 'A (GHz)', 'B (GHz)', 'C (GHz)'
] 

# ======================================================================
# QM9 Unit Conversion Factors
# ======================================================================
def get_qm9_conversions_tensor(device):
    return torch.tensor([
        1.0,                # 0 - μ (D)     
        Bohr**3 / Ang**3,   # 1 - α (Bohr³ → Å³)
        Hartree / eV,       # 2 - ε_HOMO
        Hartree / eV,       # 3 - ε_LUMO
        Hartree / eV,       # 4 - Δε
        Bohr**2 / Ang**2,   # 5 - ⟨R²⟩
        Hartree / eV,       # 6 - ZPVE
        Hartree / eV,       # 7 - U₀
        Hartree / eV,       # 8 - U
        Hartree / eV,       # 9 - H
        Hartree / eV,       # 10 - G
        1.0,                # 11 - c_v
        1.0,                # 12 - U₀_atom
        Hartree / eV,       # 13 - U_atom
        Hartree / eV,       # 14 - H_atom
        Hartree / eV,       # 15 - G_atom
        1.0,                # 16 - A (GHz)
        1.0,                # 17 - B (GHz)
        1.0                 # 18 - C (GHz)
    ], dtype=torch.float, device=device)

In [6]:
#========================================================================
# Normalize all QM9 targets
#========================================================================
y_raw_all = dataset.data.y.clone().cpu()                 # shape [N, 19]
conversions_cpu = get_qm9_conversions_tensor('cpu')      # conversion tensor [19]
y_conv_all = y_raw_all * conversions_cpu.unsqueeze(0)    # apply conversion

norm_stats = {'mean': [], 'std': []}
y_norm_all = torch.zeros_like(y_conv_all)

for i in range(y_conv_all.shape[1]):
    train_y_cpu = y_conv_all[train_idx.cpu(), i]
    mean_i = float(train_y_cpu.mean().item())
    std_i = float(train_y_cpu.std().item()) if train_y_cpu.std().item() != 0 else 1.0
    y_norm_all[:, i] = (y_conv_all[:, i] - mean_i) / std_i
    norm_stats['mean'].append(mean_i)
    norm_stats['std'].append(std_i)

dataset.data.y = y_norm_all.to(torch.float)
print("Normalization complete for all targets.")

Normalization complete for all targets.


/home/ismail/dig_envi/lib/python3.9/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [7]:
# ===========================================================================
# Graph Isomorphism Network (GIN) Model
# ===========================================================================
# Number of input features per node (atom) in each molecular graph
in_channels = dataset.num_node_features
print("Node feature dim:", in_channels)

#--------------------------
# Define GIN-Model
#--------------------------
class GINModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3, target_dim=19):
        """
        GIN model adapted to match GCN hidden size and dropout.

        Args:
            in_dim (int): Dimension of node features
            hidden_dim (int): Hidden dimension for GIN layers
            dropout (float): Dropout probability
            target_dim (int): Number of output targets
        """
        super().__init__()

        # -------------------------
        # MLPs for each GINConv layer
        # -------------------------
        mlp1 = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        mlp3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )

        # -------------------------
        # GIN convolution layers
        # -------------------------
        self.conv1 = GINConv(mlp1, train_eps=True)
        self.conv2 = GINConv(mlp2, train_eps=True)
        self.conv3 = GINConv(mlp3, train_eps=True)

        # -------------------------
        # Fully connected layers for prediction
        # -------------------------
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, target_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, batch):
        """
        
        Forward pass of the GIN.

        Args:
            batch: a PyTorch Geometric batch containing:
                    - x: node features[num_nodes, in_dim]
                    - edge_index: edge list[2, num_edges]
                    - batch: batch vector mapping nodes to molecules

        Returns :
            Predicted target values for each molecule [batch_size, 1]
            
        """
        x, edge_index, batch_vec = batch.x, batch.edge_index, batch.batch

        # Apply three GCN layers with ReLU activation
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        
        # Global mean Pooling
        x = global_mean_pool(x, batch_vec)   # The global mean pool ing takes the mean of all atom embeddings in each molecule´s graph "one feat vector per molecule"
        
        # Decoder : it transfomrs the pooled molecular embedding into the predicted target value 
        x = F.relu(self.fc1(x)) 
        x = self.dropout(x)
        return self.fc2(x)

Node feature dim: 11


In [8]:
#==============================================================================================
# Trainig Loop
#==============================================================================================
def main():
    #------------------------------
    # Hyperparameters
    #------------------------------
    epochs = 1000                             # number of training epochs
    batch_size = 128                        # batch size for training
    vt_batch_size = 256                     # batch size for validation
    lr = 1e-5                               # learning rate
    lr_decay_step = 50                      # steps after which LR is decayed
    lr_decay_factor = 0.5                   # factor to decay learning rate
    weight_decay = 1e-4                     # L2 regularization
    save_dir = 'checkpoints_GIN'            # directory to save model checkpoints 
    log_dir = 'logs_GIN'                    # TensorBoard logs directory

    #Create directories if they don't exist
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    # TensorBoard write for visualization of metrics
    writer = SummaryWriter(log_dir=log_dir)

    #-----------------------------
    # Dataloaders 
    #-----------------------------

    # Shuffled batches for training; sequential batches for validation/testing
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(valid_dataset, batch_size=vt_batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=vt_batch_size, shuffle=False)

    #----------------------------------------
    # Model
    #----------------------------------------
    # GIN model
    model = GINModel(in_channels).to(device)  

    #----------------------------------------
    # Optimizer and Scheduler
    #----------------------------------------
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=lr_decay_step, gamma=lr_decay_factor)

    # Track best validation/test performance
    best_mean_val=float('inf')
    best_val = [float('inf')] * len(PROPERTY_NAMES)
    best_test = [float('inf')] * len(PROPERTY_NAMES)

    # Print total number of trainabel parameters and target property
    print("#params:", sum(p.numel() for p in model.parameters()))
    print("Training for all 19 targets") 
    
    #--------------------------------------------
    # Training Loop
    #--------------------------------------------
    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch} ===")

        #Train for one epoch
        train_loss = train_epoch(model, train_loader, optimizer, device)

        # Evaluate on validation and test sets (MAE in original units)
        val_mae = evaluate(model, val_loader, device, norm_stats['mean'], norm_stats['std'])
        test_mae = evaluate(model, test_loader, device, norm_stats['mean'], norm_stats['std'])

        # Print epoch metrics
        print(f"Train loss (MSE): {train_loss:.6f}")
        for i, prop in enumerate(PROPERTY_NAMES):
            print(f"  {prop:15s} | Val MAE: {val_mae[i]:.6f} | Test MAE: {test_mae[i]:.6f}")
            writer.add_scalar(f'val_mae/{prop}', val_mae[i], epoch)
            writer.add_scalar(f'test_mae/{prop}', test_mae[i], epoch)
        #---------------------------------------------
        # Save checkpoint if validation improves 
        #---------------------------------------------
        mean_val_mae = sum(val_mae) / len(val_mae)
        
        # If mean validation MAE improved → save one best model
        if mean_val_mae < best_mean_val:
            best_mean_val = mean_val_mae
            best_val = val_mae
            best_test = test_mae
        
            save_path = os.path.join(save_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val': best_val,
                'best_test': best_test,
                'best_mean_val': best_mean_val
            }, save_path)
        
            print(f" Saved best overall model (mean validation MAE improved to {best_mean_val:.4f})")

        # Update learning rate according to scheduler
        scheduler.step()
    # Close TensorBoard writer
    writer.close()
    print("\nFinished training.")
    print("Best validation and test MAEs per property:")
    for prop, v, t in zip(PROPERTY_NAMES, best_val, best_test):
        print(f"  {prop:15s} | Best validation MAE: {v:.6f} | Test MAE at best val: {t:.6f}")

    #print("Best validation MAE:", best_val)
    #print("Test MAE at best val:", best_test)

# Run the training loop
if __name__ == "__main__":
    main()

#params: 95126
Training for all 19 targets

=== Epoch 1 ===


Train loss (MSE): 0.830131
  μ (D)           | Val MAE: 1.051107 | Test MAE: 1.071974
  α (Ang³)        | Val MAE: 0.730932 | Test MAE: 0.724362
  ε_HOMO (eV)     | Val MAE: 11.384344 | Test MAE: 11.401171
  ε_LUMO (eV)     | Val MAE: 22.189701 | Test MAE: 22.482718
  Δε (eV)         | Val MAE: 25.229719 | Test MAE: 25.317064
  ⟨R²⟩ (Ang²)     | Val MAE: 50.915939 | Test MAE: 50.735100
  ZPVE (eV)       | Val MAE: 11.400792 | Test MAE: 11.260612
  U₀ (eV)         | Val MAE: 18890.914062 | Test MAE: 18480.591797
  U (eV)          | Val MAE: 17852.308594 | Test MAE: 17456.576172
  H (eV)          | Val MAE: 19509.697266 | Test MAE: 19153.755859
  G (eV)          | Val MAE: 18199.332031 | Test MAE: 17841.876953
  c_v             | Val MAE: 2.543126 | Test MAE: 2.509670
  U₀_atom         | Val MAE: 4.417888 | Test MAE: 4.340853
  U_atom          | Val MAE: 131.666946 | Test MAE: 129.857742
  H_atom          | Val MAE: 115.945374 | Test MAE: 113.703926
  G_atom          | Val MAE: 135.36390

Train loss (MSE): 0.644431
  μ (D)           | Val MAE: 0.977774 | Test MAE: 0.997123
  α (Ang³)        | Val MAE: 0.580854 | Test MAE: 0.571073
  ε_HOMO (eV)     | Val MAE: 10.343324 | Test MAE: 10.360684
  ε_LUMO (eV)     | Val MAE: 18.464066 | Test MAE: 18.667585
  Δε (eV)         | Val MAE: 21.168348 | Test MAE: 21.195498
  ⟨R²⟩ (Ang²)     | Val MAE: 47.150433 | Test MAE: 46.945549
  ZPVE (eV)       | Val MAE: 8.095557 | Test MAE: 7.875488
  U₀ (eV)         | Val MAE: 15342.197266 | Test MAE: 14924.430664
  U (eV)          | Val MAE: 15481.265625 | Test MAE: 15058.335938
  H (eV)          | Val MAE: 16258.413086 | Test MAE: 15893.564453
  G (eV)          | Val MAE: 15139.817383 | Test MAE: 14755.382812
  c_v             | Val MAE: 2.190969 | Test MAE: 2.148844
  U₀_atom         | Val MAE: 3.468099 | Test MAE: 3.370022
  U_atom          | Val MAE: 94.475494 | Test MAE: 91.660347
  H_atom          | Val MAE: 90.131897 | Test MAE: 87.130600
  G_atom          | Val MAE: 96.905983 | Tes

Train loss (MSE): 0.568177
  μ (D)           | Val MAE: 0.935827 | Test MAE: 0.952600
  α (Ang³)        | Val MAE: 0.519684 | Test MAE: 0.508448
  ε_HOMO (eV)     | Val MAE: 9.160144 | Test MAE: 9.193829
  ε_LUMO (eV)     | Val MAE: 16.079544 | Test MAE: 16.188873
  Δε (eV)         | Val MAE: 18.106909 | Test MAE: 18.120451
  ⟨R²⟩ (Ang²)     | Val MAE: 44.105415 | Test MAE: 43.885017
  ZPVE (eV)       | Val MAE: 7.057872 | Test MAE: 6.856170
  U₀ (eV)         | Val MAE: 13761.320312 | Test MAE: 13352.197266
  U (eV)          | Val MAE: 14046.501953 | Test MAE: 13603.126953
  H (eV)          | Val MAE: 14590.396484 | Test MAE: 14206.726562
  G (eV)          | Val MAE: 13915.870117 | Test MAE: 13516.122070
  c_v             | Val MAE: 2.014599 | Test MAE: 1.973227
  U₀_atom         | Val MAE: 3.346511 | Test MAE: 3.227300
  U_atom          | Val MAE: 90.061539 | Test MAE: 86.889671
  H_atom          | Val MAE: 91.108429 | Test MAE: 87.922874
  G_atom          | Val MAE: 86.938133 | Test 

Train loss (MSE): 0.531158
  μ (D)           | Val MAE: 0.909947 | Test MAE: 0.925401
  α (Ang³)        | Val MAE: 0.503105 | Test MAE: 0.491922
  ε_HOMO (eV)     | Val MAE: 8.248498 | Test MAE: 8.302392
  ε_LUMO (eV)     | Val MAE: 14.434441 | Test MAE: 14.503514
  Δε (eV)         | Val MAE: 16.182190 | Test MAE: 16.179758
  ⟨R²⟩ (Ang²)     | Val MAE: 41.590092 | Test MAE: 41.270100
  ZPVE (eV)       | Val MAE: 6.620622 | Test MAE: 6.412999
  U₀ (eV)         | Val MAE: 12811.288086 | Test MAE: 12394.511719
  U (eV)          | Val MAE: 13154.788086 | Test MAE: 12697.013672
  H (eV)          | Val MAE: 13461.113281 | Test MAE: 13067.081055
  G (eV)          | Val MAE: 13073.593750 | Test MAE: 12660.085938
  c_v             | Val MAE: 1.893573 | Test MAE: 1.850567
  U₀_atom         | Val MAE: 3.345315 | Test MAE: 3.225233
  U_atom          | Val MAE: 90.287834 | Test MAE: 87.168259
  H_atom          | Val MAE: 92.208611 | Test MAE: 89.134377
  G_atom          | Val MAE: 85.516312 | Test 

...

Train loss (MSE): 0.303548
  μ (D)           | Val MAE: 0.681864 | Test MAE: 0.689973
  α (Ang³)        | Val MAE: 0.409476 | Test MAE: 0.402194
  ε_HOMO (eV)     | Val MAE: 5.073726 | Test MAE: 5.185859
  ε_LUMO (eV)     | Val MAE: 6.879391 | Test MAE: 6.895738
  Δε (eV)         | Val MAE: 8.351906 | Test MAE: 8.375171
  ⟨R²⟩ (Ang²)     | Val MAE: 29.099100 | Test MAE: 28.777979
  ZPVE (eV)       | Val MAE: 4.947877 | Test MAE: 4.829001
  U₀ (eV)         | Val MAE: 10079.886719 | Test MAE: 9762.309570
  U (eV)          | Val MAE: 10196.551758 | Test MAE: 9882.514648
  H (eV)          | Val MAE: 10147.143555 | Test MAE: 9824.297852
  G (eV)          | Val MAE: 10197.879883 | Test MAE: 9887.130859
  c_v             | Val MAE: 1.332634 | Test MAE: 1.302772
  U₀_atom         | Val MAE: 2.668596 | Test MAE: 2.596401
  U_atom          | Val MAE: 72.859344 | Test MAE: 70.919754
  H_atom          | Val MAE: 73.220222 | Test MAE: 71.167336
  G_atom          | Val MAE: 67.661331 | Test MAE: 65.

Train loss (MSE): 0.315771
  μ (D)           | Val MAE: 0.681079 | Test MAE: 0.688827
  α (Ang³)        | Val MAE: 0.420504 | Test MAE: 0.412915
  ε_HOMO (eV)     | Val MAE: 5.047362 | Test MAE: 5.163964
  ε_LUMO (eV)     | Val MAE: 6.817189 | Test MAE: 6.788362
  Δε (eV)         | Val MAE: 8.291163 | Test MAE: 8.291004
  ⟨R²⟩ (Ang²)     | Val MAE: 29.046267 | Test MAE: 28.758734
  ZPVE (eV)       | Val MAE: 4.939933 | Test MAE: 4.812230
  U₀ (eV)         | Val MAE: 9737.104492 | Test MAE: 9393.150391
  U (eV)          | Val MAE: 9836.127930 | Test MAE: 9492.881836
  H (eV)          | Val MAE: 9812.438477 | Test MAE: 9461.532227
  G (eV)          | Val MAE: 9828.129883 | Test MAE: 9489.232422
  c_v             | Val MAE: 1.345295 | Test MAE: 1.315704
  U₀_atom         | Val MAE: 2.731785 | Test MAE: 2.659060
  U_atom          | Val MAE: 74.663528 | Test MAE: 72.692390
  H_atom          | Val MAE: 74.968582 | Test MAE: 72.899857
  G_atom          | Val MAE: 69.425812 | Test MAE: 67.6786

Train loss (MSE): 0.301530
  μ (D)           | Val MAE: 0.682149 | Test MAE: 0.689737
  α (Ang³)        | Val MAE: 0.427461 | Test MAE: 0.419541
  ε_HOMO (eV)     | Val MAE: 5.038602 | Test MAE: 5.158230
  ε_LUMO (eV)     | Val MAE: 6.816381 | Test MAE: 6.822769
  Δε (eV)         | Val MAE: 8.286658 | Test MAE: 8.327206
  ⟨R²⟩ (Ang²)     | Val MAE: 29.205334 | Test MAE: 28.937510
  ZPVE (eV)       | Val MAE: 5.010152 | Test MAE: 4.876610
  U₀ (eV)         | Val MAE: 9708.280273 | Test MAE: 9359.959961
  U (eV)          | Val MAE: 9813.630859 | Test MAE: 9465.798828
  H (eV)          | Val MAE: 9774.144531 | Test MAE: 9420.800781
  G (eV)          | Val MAE: 9793.743164 | Test MAE: 9451.521484
  c_v             | Val MAE: 1.356283 | Test MAE: 1.323897
  U₀_atom         | Val MAE: 2.728467 | Test MAE: 2.651919
  U_atom          | Val MAE: 74.541222 | Test MAE: 72.456314
  H_atom          | Val MAE: 74.860817 | Test MAE: 72.686081
  G_atom          | Val MAE: 69.335304 | Test MAE: 67.4720

Train loss (MSE): 0.308372
  μ (D)           | Val MAE: 0.679838 | Test MAE: 0.688392
  α (Ang³)        | Val MAE: 0.415941 | Test MAE: 0.408737
  ε_HOMO (eV)     | Val MAE: 5.044579 | Test MAE: 5.170005
  ε_LUMO (eV)     | Val MAE: 6.907405 | Test MAE: 6.862291
  Δε (eV)         | Val MAE: 8.359836 | Test MAE: 8.349732
  ⟨R²⟩ (Ang²)     | Val MAE: 29.114920 | Test MAE: 28.829321
  ZPVE (eV)       | Val MAE: 4.970027 | Test MAE: 4.847633
  U₀ (eV)         | Val MAE: 9990.715820 | Test MAE: 9662.498047
  U (eV)          | Val MAE: 10102.481445 | Test MAE: 9781.534180
  H (eV)          | Val MAE: 10068.015625 | Test MAE: 9737.347656
  G (eV)          | Val MAE: 10106.210938 | Test MAE: 9785.538086
  c_v             | Val MAE: 1.356115 | Test MAE: 1.325951
  U₀_atom         | Val MAE: 2.737644 | Test MAE: 2.667491
  U_atom          | Val MAE: 74.799377 | Test MAE: 72.886238
  H_atom          | Val MAE: 75.121338 | Test MAE: 73.117882
  G_atom          | Val MAE: 69.541595 | Test MAE: 67.8

Train loss (MSE): 0.309944
  μ (D)           | Val MAE: 0.680437 | Test MAE: 0.688619
  α (Ang³)        | Val MAE: 0.409470 | Test MAE: 0.401692
  ε_HOMO (eV)     | Val MAE: 5.065416 | Test MAE: 5.182900
  ε_LUMO (eV)     | Val MAE: 6.833480 | Test MAE: 6.812245
  Δε (eV)         | Val MAE: 8.331461 | Test MAE: 8.334198
  ⟨R²⟩ (Ang²)     | Val MAE: 28.978384 | Test MAE: 28.623323
  ZPVE (eV)       | Val MAE: 4.966971 | Test MAE: 4.840450
  U₀ (eV)         | Val MAE: 10268.041992 | Test MAE: 9948.289062
  U (eV)          | Val MAE: 10392.597656 | Test MAE: 10079.074219
  H (eV)          | Val MAE: 10335.734375 | Test MAE: 10014.555664
  G (eV)          | Val MAE: 10395.674805 | Test MAE: 10086.378906
  c_v             | Val MAE: 1.349969 | Test MAE: 1.319056
  U₀_atom         | Val MAE: 2.686279 | Test MAE: 2.611849
  U_atom          | Val MAE: 73.343140 | Test MAE: 71.329872
  H_atom          | Val MAE: 73.744247 | Test MAE: 71.640053
  G_atom          | Val MAE: 68.161324 | Test MAE: 